<a href="https://colab.research.google.com/github/SOUMYADEEP-rgb/Crtic_cot/blob/main/data_mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch transformers accelerate datasets tqdm sympy pylatexenc scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=d59c5c52fd2ffc80505271f1befe6103829e6053763fe7d2b0a950cfd52ebe86
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
from datasets import load_dataset

dataset = load_dataset("gsm8k", "main")

train = dataset["train"]
test = dataset["test"]

In [ ]:
import json

def convert(split, filename, limit):
    with open(filename, "w") as f:
        for i, item in enumerate(split):
            if i >= limit:
                break
            json.dump({
                "problem": item["question"],
                "answer": item["answer"].split("####")[-1].strip()
            }, f)
            f.write("\n")

convert(train, "train.jsonl", 200)   # keep small
convert(test, "test.jsonl", 50)

In [ ]:
!head train.jsonl

{"problem": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?", "answer": "72"}
{"problem": "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?", "answer": "10"}
{"problem": "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?", "answer": "5"}
{"problem": "Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?", "answer": "42"}
{"problem": "James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year?", "answer

In [ ]:
!python run_math_vllm.py --mode solve --src train.jsonl --tgt solve.jsonl --max_instance 10

Loading weights: 100% 453/453 [00:01<00:00, 254.08it/s, Materializing param=model.layers.31.self_attn.v_proj.weight]
100% 10/10 [01:10<00:00,  7.03s/it]


In [ ]:
!head solve.jsonl

{"problem": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?", "answer": "72", "pre_generated_steps": [["\\begin{align*}\n\\text{Total clips sold in April and May} &= \\text{Clips sold in April} + \\text{Clips sold in May} \\\\\n&= 48 + \\frac{48}{2} \\\\\n&= 48 + 24 \\\\\n&= 72\n\\end{align*}"]], "predict_answer": ["72"], "correct": [true]}
{"problem": "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?", "answer": "10", "pre_generated_steps": [["Step 1: Convert 50 minutes to hours.", "Step 2: Multiply the hours by the hourly rate.", "Step 3: Round the answer to the nearest cent."]], "predict_answer": ["10"], "correct": [true]}
{"problem": "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twi

In [ ]:
!python run_math_vllm.py --mode solve --src train.jsonl --tgt solve.jsonl --max_instance 50

Loading weights: 100% 453/453 [00:01<00:00, 255.05it/s, Materializing param=model.layers.31.self_attn.v_proj.weight]
100% 50/50 [10:44<00:00, 12.90s/it]


In [ ]:
!python run_math_vllm.py --mode solve --src train.jsonl --tgt solve.jsonl

Loading weights: 100% 453/453 [00:01<00:00, 245.36it/s, Materializing param=model.layers.31.self_attn.v_proj.weight]
100% 100/100 [12:13<00:00,  7.34s/it]


In [ ]:
!python run_math_vllm.py --mode critic --src solve.jsonl --tgt critic.jsonl

Loading weights: 100% 453/453 [00:01<00:00, 246.64it/s, Materializing param=model.layers.31.self_attn.v_proj.weight]
100% 10/10 [02:22<00:00, 14.30s/it]


In [ ]:
!python run_math_vllm.py --mode refine --src critic.jsonl --tgt refine.jsonl

Loading weights: 100% 453/453 [00:01<00:00, 260.02it/s, Materializing param=model.layers.31.self_attn.v_proj.weight]
100% 100/100 [11:17<00:00,  6.78s/it]


In [ ]:
!python run_math_vllm.py --mode refine --src critic.jsonl --tgt refine.jsonl

Loading weights: 100% 453/453 [00:01<00:00, 257.61it/s, Materializing param=model.layers.31.self_attn.v_proj.weight]
100% 10/10 [02:21<00:00, 14.18s/it]


In [ ]:
!python eval_math_critic.py

Accuracy: 60.0
